In [2]:
import pandas as pd

In [4]:
import requests

url = "https://donnees.hydroquebec.com/api/explore/v2.1/catalog/datasets/demande-electricite-quebec/records"
params = {
    "limit": 100,
    "order_by": "-date"
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
#print(r.json())

df  = pd.DataFrame(r.json()['results'])

df.tail(10)

,date,valeurs_demandetotal
90,2026-03-13T05:15:00+00:00,26557.0
91,2026-03-13T05:00:00+00:00,26606.0
92,2026-03-13T04:45:00+00:00,26461.0
93,2026-03-13T04:30:00+00:00,26721.0
94,2026-03-13T04:15:00+00:00,26959.0
95,2026-03-13T04:00:00+00:00,27124.0
96,2026-03-13T03:45:00+00:00,27164.0
97,2026-03-13T03:30:00+00:00,27535.0
98,2026-03-13T03:15:00+00:00,27589.0
99,2026-03-13T03:00:00+00:00,28094.0


In [31]:
import requests
import pandas as pd

url = "https://donnees.hydroquebec.com/api/explore/v2.1/catalog/datasets/demande-electricite-quebec/records"

params = {
    "limit": 100,
    "where": "date >= date'2026-03-13T03:00:00Z' and date < date'2025-06-13T05:00:00Z'",
    "order_by": "date"
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()

data = r.json()["results"]
df = pd.DataFrame(data)

if not df.empty:
    df["date"] = pd.to_datetime(df["date"], utc=True)

print(df.head())
print(df.tail())
print(df.shape)

Empty DataFrame
Columns: []
Index: []
Empty DataFrame
Columns: []
Index: []
(0, 0)


In [32]:
import requests
import pandas as pd


BASE_URL = "https://donnees.hydroquebec.com/api/explore/v2.1/catalog/datasets/demande-electricite-quebec/records"


def fetch_hydro_quebec_between_dates(
    start_date: str,
    end_date: str,
    batch_size: int = 100,
) -> pd.DataFrame:
    """
    Récupère toutes les données Hydro-Québec entre deux dates inclusives/exclusives.
    
    Paramètres
    ----------
    start_date : str
        Date de début au format ISO, ex: "2025-02-12T00:00:00Z"
    end_date : str
        Date de fin exclusive au format ISO, ex: "2025-02-14T00:00:00Z"
    batch_size : int
        Nombre de lignes par appel API
        
    Retour
    ------
    pd.DataFrame
    """
    all_rows = []
    offset = 0

    while True:
        params = {
            "limit": batch_size,
            "offset": offset,
            "where": f"date >= date'{start_date}' and date < date'{end_date}'",
            "order_by": "date"
        }

        r = requests.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()

        results = r.json().get("results", [])
        
        if not results:
            break

        all_rows.extend(results)
        print(f"Batch récupéré: {len(results)} lignes | Total: {len(all_rows)}")

        if len(results) < batch_size:
            break

        offset += batch_size

    df = pd.DataFrame(all_rows)

    if not df.empty and "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], utc=True)

    return df


# Exemple : du 12 février 2025 au 13 février 2025 inclus
df = fetch_hydro_quebec_between_dates(
    start_date="2025-02-12T00:00:00Z",
    end_date="2025-02-14T00:00:00Z",  # fin exclusive
    batch_size=100
)

print(df.head())
print(df.tail())
print(df.shape)

Empty DataFrame
Columns: []
Index: []
Empty DataFrame
Columns: []
Index: []
(0, 0)
